# 00 · Descarga de datos

Baja todas las fuentes del TPO y las deja en `data/raw/`:

| Fuente | Origen |
|---|---|
| Cajeros automáticos, barrios, bocas de subte, usos del suelo 2022-2024, molinetes 2025 y 2026 | BA Data (GCBA) |
| Censo 2022 por radio censal | INDEC, vía `censoargentino` (Hugging Face) |
| Estaciones, bancos, cajeros de todas las redes, comercios | OpenStreetMap |

**Cómo usarlo en Colab:** *Entorno de ejecución → Ejecutar todo*. Tarda entre 5 y 15 minutos.
Al final se descarga **`datos_tpo.zip`** con todo lo necesario (los archivos pesados van resumidos).

BA Data a veces devuelve errores de servidor: cada archivo se reintenta solo, y si alguno falla igual el resto sigue.
Si querés reintentar lo que falló, volvé a ejecutar todo: lo que ya bajó no se descarga de nuevo.

In [ ]:
# Preparar el entorno: en Colab clona el repo e instala lo que falta
import subprocess, sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/CienciaDeDatos")
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/ValentinoDiaz0509/CienciaDeDatos.git", str(RAIZ)], check=True)
    else:
        subprocess.run(["git", "-C", str(RAIZ), "pull", "-q"], check=False)  # traer la última versión
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "censoargentino[geo]"], check=True)
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(RAIZ))
from src import descarga as d
print("Los datos van a:", d.DIR_RAW)

## 1. BA Data
Cajeros, barrios, bocas de subte, usos del suelo y molinetes. Los dos últimos son pesados (el shapefile de usos del suelo y los molinetes pueden tardar varios minutos); para saltearlos, cambiá `incluir_pesadas` a `False`.

In [ ]:
d.descargar_fuentes_ba(incluir_pesadas=True)

## 2. Censo 2022 por radio censal
Población, edad, condición de actividad, nivel educativo, NBI, privación material y acceso a internet/celular/computadora, para los ~3.800 radios censales de CABA. Además baja los polígonos de los radios (un archivo de todo el país de ~58 MB) y se queda con los de CABA.

In [ ]:
d.descargar_censo()

## 3. OpenStreetMap
Estaciones de subte y tren, sucursales bancarias, cajeros de todas las redes y comercios.

In [ ]:
d.descargar_osm()

## 4. Resumir los archivos pesados
Los molinetes se resumen a pasajeros promedio por estación, hora y tipo de día. Los usos del suelo pasan a Parquet (pesa varias veces menos) y, a partir del shapefile, se les agrega un punto (lon, lat) por parcela, porque el CSV oficial no trae coordenadas.

In [ ]:
d.procesar_pesados()

## 5. Resumen y zip
Al terminar se descarga `datos_tpo.zip`: ese es el archivo para compartir. Si algún paso dice **NO**, el zip igual sirve con lo que sí bajó, y adentro va `estado_descarga.json` con el detalle de cada error.

In [ ]:
d.resumen()
zip_path = d.empaquetar()

if EN_COLAB:
    from google.colab import files
    files.download(str(zip_path))

*Opcional:* para guardar el zip también en tu Google Drive, descomentá y ejecutá la celda de abajo.

In [ ]:
# from google.colab import drive
# import shutil
# drive.mount("/content/drive")
# shutil.copy(zip_path, "/content/drive/MyDrive/datos_tpo.zip")